# GLN 模型展示教程

## 概述

本 notebook 深入剖析 **GLN (Graph Logic Network)** 的模型计算架构和推理原理。

GLN 的核心思想是将逆合成预测分解为一个层次化的条件概率推理过程：

$$P(\text{reactants} | \text{product}) = \sum_{c, t} P(c | p) \cdot P(t | c, p) \cdot P(r | t, p)$$

其中：
- $P(c|p)$：给定产物 $p$，预测反应中心 $c$（**CenterProbCalc**）
- $P(t|c,p)$：给定产物和中心，预测模板 $t$（**ActiveProbCalc**）
- $P(r|t,p)$：给定产物和模板，验证反应物 $r$（**ReactionProbCalc**）

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 分子图的特征编码（节点/边特征） |
| 2 | 图神经网络 (GNN) 编码器 |
| 3 | 三个概率计算模块 |
| 4 | Jagged Softmax 变长序列归一化 |
| 5 | 完整的 GraphPath 模型 |
| 6 | 推理流程的端到端演示 |
| 7 | 训练流程概览 |

> **说明**: 本教程用最小化的纯 Python/PyTorch 代码重现 GLN 的核心计算逻辑，**不依赖 GLN 原始代码的 import**，所有模块均面向教学设计，代码经过简化处理，仅用于教学演示，不建议直接用于追求论文级性能的生产训练。

---

In [1]:
# ====== 基础导入 ======
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_scatter import scatter_add, scatter_mean
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from collections import defaultdict
from IPython.display import HTML, display

# 检查设备
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'计算设备: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'所有导入成功！')

计算设备: cpu
PyTorch: 2.10.0+cpu
所有导入成功！


---

## 1. 分子图的特征编码

GLN 将每个分子表示为图 $G = (V, E)$，其中原子是节点，化学键是边。

### 节点特征编码

每个原子的特征被 **位打包 (bit packing)** 为一个 32-bit 整数，以节省内存和加速处理：

```
原子特征 (int32):
┌──────────────┬────────────┬─────────┬──────────┬──────────────┐
│ 芳香性(1 bit) │ 化合价(4 bit) │ 氢数(4 bit) │ 度数(4 bit) │ 原子类型(8 bit) │
└──────────────┴────────────┴─────────┴──────────┴──────────────┘
```

GLN 的 C++ 扩展库 (MGLIB) 负责将打包整数展开为 one-hot 向量，供 GNN 使用。

### GLN 源码对应
- `gln/mods/mol_gnn/mol_utils.py` → `get_atom_feat()`, `get_bond_feat()`, `MolGraph`

In [2]:
# ====== 1.1 分子图特征编码 ======
# 对应 GLN 源码: gln/mods/mol_gnn/mol_utils.py

# GLN 使用的原子类型 (最常见的有机原子)
ATOM_LIST = [6, 7, 8, 9, 15, 16, 17, 35, 53]  # C, N, O, F, P, S, Cl, Br, I
ATOM_IDX_MAP = {a: i for i, a in enumerate(ATOM_LIST)}

# ====== 教学版本：用 one-hot 特征取代位打包 ======
# GLN 原始代码使用 C++ 扩展 (MGLIB) 将打包整数展开为 one-hot
# 这里我们直接生成 one-hot 向量，效果等价

def get_atom_features(atom, sanitized=True):
    """
    生成原子特征向量（教学版，使用 one-hot 编码）。
    
    GLN 原始代码在 C++ 中将打包整数展开为 one-hot 向量。
    这里我们直接生成 one-hot，功能等价。
    
    特征维度: 原子类型(10) + 度数(6) + 氢数(6) + 化合价(7) + 芳香性(2) = 31
    
    注意: 对于 SMARTS 查询分子 (sanitized=False)，RDKit 2025+ 不允许直接
    调用 GetImplicitValence() / GetTotalNumHs()，因为查询分子未计算化合价。
    GLN 源码对这两项在 SMARTS 模式下使用特殊标记值。
    """
    features = []
    
    # 原子类型 one-hot (10维: 9种原子 + 未知)
    atom_type = [0] * (len(ATOM_LIST) + 1)
    atomic_num = atom.GetAtomicNum()
    if atomic_num in ATOM_IDX_MAP:
        atom_type[ATOM_IDX_MAP[atomic_num]] = 1
    else:
        atom_type[-1] = 1  # 未知原子
    features.extend(atom_type)
    
    # 度数 one-hot (6维: 0-5)
    degree = [0] * 6
    degree[min(atom.GetDegree(), 5)] = 1
    features.extend(degree)
    
    # 氢原子数 one-hot (6维: 0-5)
    h_count = [0] * 6
    if sanitized:
        h_count[min(atom.GetTotalNumHs(), 5)] = 1
    else:
        h_count[5] = 1  # SMARTS 模式的特殊标记
    features.extend(h_count)
    
    # 隐式化合价 one-hot (7维: 0-6)
    # SMARTS 查询分子在新版 RDKit 中无法调用 GetImplicitValence()
    valence = [0] * 7
    if sanitized:
        valence[min(atom.GetImplicitValence(), 6)] = 1
    else:
        valence[6] = 1  # SMARTS 模式的特殊标记（溢出值）
    features.extend(valence)
    
    # 芳香性 (2维)
    aromatic = [0, 0]
    aromatic[int(atom.GetIsAromatic())] = 1
    features.extend(aromatic)
    
    return features

NUM_NODE_FEATS = 31  # 10 + 6 + 6 + 7 + 2


def get_bond_features(bond, sanitized=True):
    """
    生成化学键特征向量。
    
    特征维度: 键类型(4) + 共轭(2) + 环内(3) = 9
    """
    features = []
    
    # 键类型 one-hot (4维)
    bt_map = {
        Chem.rdchem.BondType.SINGLE: 0,
        Chem.rdchem.BondType.DOUBLE: 1,
        Chem.rdchem.BondType.TRIPLE: 2,
        Chem.rdchem.BondType.AROMATIC: 3,
    }
    bond_type = [0] * 4
    bond_type[bt_map.get(bond.GetBondType(), 0)] = 1
    features.extend(bond_type)
    
    # 共轭性 (2维)
    conj = [0, 0]
    conj[int(bond.GetIsConjugated())] = 1
    features.extend(conj)
    
    # 是否在环内 (3维: 否/是/未知)
    ring = [0, 0, 0]
    if sanitized:
        in_ring = bond.GetOwningMol().GetRingInfo().NumBondRings(bond.GetIdx()) > 0
        ring[int(in_ring)] = 1
    else:
        ring[2] = 1  # SMARTS 模式未知
    features.extend(ring)
    
    return features

NUM_EDGE_FEATS = 9  # 4 + 2 + 3


# ====== 演示 ======
demo_mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1')  # 乙酸苯酯
print(f'演示分子: 乙酸苯酯 CC(=O)Oc1ccccc1')
print(f'  原子数: {demo_mol.GetNumAtoms()}')
print(f'  键数: {demo_mol.GetNumBonds()}')
print(f'  节点特征维度: {NUM_NODE_FEATS}')
print(f'  边特征维度: {NUM_EDGE_FEATS}')
print()

# 展示一个原子的特征
atom = demo_mol.GetAtomWithIdx(0)  # 第一个碳
feat = get_atom_features(atom)
print(f'原子 0 ({atom.GetSymbol()}) 特征向量 (维度={len(feat)}):')
print(f'  原子类型: {feat[:10]}  (C=1)')
print(f'  度数:     {feat[10:16]}  (degree={atom.GetDegree()})')
print(f'  氢数:     {feat[16:22]}  (H={atom.GetTotalNumHs()})')
print(f'  化合价:   {feat[22:29]}  (valence={atom.GetImplicitValence()})')
print(f'  芳香性:   {feat[29:31]}  (aromatic={atom.GetIsAromatic()})')

演示分子: 乙酸苯酯 CC(=O)Oc1ccccc1
  原子数: 10
  键数: 10
  节点特征维度: 31
  边特征维度: 9

原子 0 (C) 特征向量 (维度=31):
  原子类型: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]  (C=1)
  度数:     [0, 1, 0, 0, 0, 0]  (degree=1)
  氢数:     [0, 0, 0, 1, 0, 0]  (H=3)
  化合价:   [0, 0, 0, 1, 0, 0, 0]  (valence=3)
  芳香性:   [1, 0]  (aromatic=False)


[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead


In [3]:
# ====== 1.2 构建分子图的 PyTorch 表示 ======
# 对应 GLN 源码: gln/mods/mol_gnn/mol_utils.py → MolGraph, SmilesMols

class MolGraph:
    """
    分子的图表示，包含节点特征、边特征和拓扑结构。
    
    对应 GLN 源码: gln/mods/mol_gnn/mol_utils.py → MolGraph
    """
    def __init__(self, name, mol, sanitized=True):
        self.name = name
        self.num_nodes = mol.GetNumAtoms()
        self.num_edges = mol.GetNumBonds()
        
        # 节点特征矩阵 [num_nodes, node_feat_dim]
        self.node_feats = np.array(
            [get_atom_features(a, sanitized) for a in mol.GetAtoms()],
            dtype=np.float32
        )
        
        # 边特征矩阵 [num_edges, edge_feat_dim]
        self.bond_feats = np.array(
            [get_bond_features(b, sanitized) for b in mol.GetBonds()],
            dtype=np.float32
        ) if self.num_edges > 0 else np.zeros((0, NUM_EDGE_FEATS), dtype=np.float32)
        
        # 边索引 (双向): [2, 2*num_edges]
        edge_from = []
        edge_to = []
        for bond in mol.GetBonds():
            i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            # 双向边
            edge_from.extend([i, j])
            edge_to.extend([j, i])
        self.edge_from = np.array(edge_from, dtype=np.int64)
        self.edge_to = np.array(edge_to, dtype=np.int64)
        
        # 双向边特征（每条边重复两次）
        if self.num_edges > 0:
            self.edge_feats_bidir = np.repeat(self.bond_feats, 2, axis=0)
        else:
            self.edge_feats_bidir = np.zeros((0, NUM_EDGE_FEATS), dtype=np.float32)


class MolGraphPool:
    """
    分子图缓存池，避免重复构图。
    对应 GLN 源码: mol_utils.py → SmilesMols / SmartsMols
    """
    def __init__(self, sanitized=True):
        self.sanitized = sanitized
        self.cache = {}
    
    def get_mol_graph(self, name):
        if name is None or len(name.strip()) == 0:
            return None
        if name not in self.cache:
            mol = Chem.MolFromSmiles(name) if self.sanitized else Chem.MolFromSmarts(name)
            if mol is None:
                return None
            self.cache[name] = MolGraph(name, mol, self.sanitized)
        return self.cache[name]


# 初始化全局缓存
SmilesMols = MolGraphPool(sanitized=True)
SmartsMols = MolGraphPool(sanitized=False)


def prepare_batch(graph_list):
    """
    将多个分子图合并为一个批次 (batch)。
    
    对应 GLN 源码: gln/mods/mol_gnn/gnn_family/utils.py → prepare_gnn()
    GLN 原始代码通过 C++ 扩展 MGLIB 实现，这里用纯 Python 等价实现。
    
    合并方式: 
    - 节点特征: 拼接所有图的节点特征
    - 边特征: 拼接所有图的边特征
    - 边索引: 偏移后拼接（避免不同图节点混淆）
    - g_idx: 标识每个节点/边属于哪个图
    """
    all_node_feats = []
    all_edge_feats = []
    all_edge_from = []
    all_edge_to = []
    g_idx = []  # 每个节点所属的图索引
    
    offset = 0
    for i, g in enumerate(graph_list):
        all_node_feats.append(g.node_feats)
        all_edge_feats.append(g.edge_feats_bidir)
        all_edge_from.append(g.edge_from + offset)
        all_edge_to.append(g.edge_to + offset)
        g_idx.extend([i] * g.num_nodes)
        offset += g.num_nodes
    
    node_feat = torch.FloatTensor(np.concatenate(all_node_feats, axis=0)).to(DEVICE)
    edge_feat = torch.FloatTensor(np.concatenate(all_edge_feats, axis=0)).to(DEVICE) if sum(g.num_edges for g in graph_list) > 0 else None
    edge_from = torch.LongTensor(np.concatenate(all_edge_from)).to(DEVICE)
    edge_to = torch.LongTensor(np.concatenate(all_edge_to)).to(DEVICE)
    g_idx = torch.LongTensor(g_idx).to(DEVICE)
    
    return node_feat, edge_feat, edge_from, edge_to, g_idx


# ====== 演示 ======
g1 = SmilesMols.get_mol_graph('CC(=O)O')     # 乙酸
g2 = SmilesMols.get_mol_graph('c1ccccc1')     # 苯
g3 = SmilesMols.get_mol_graph('CC(=O)Oc1ccccc1')  # 乙酸苯酯

node_feat, edge_feat, edge_from, edge_to, g_idx = prepare_batch([g1, g2, g3])

print('批次合并结果:')
print(f'  乙酸: {g1.num_nodes} 原子, {g1.num_edges} 键')
print(f'  苯: {g2.num_nodes} 原子, {g2.num_edges} 键')
print(f'  乙酸苯酯: {g3.num_nodes} 原子, {g3.num_edges} 键')
print(f'  合并后节点特征: {node_feat.shape}')
print(f'  合并后边特征: {edge_feat.shape if edge_feat is not None else "N/A"}')
print(f'  节点所属图: {g_idx.tolist()}')

批次合并结果:
  乙酸: 4 原子, 3 键
  苯: 6 原子, 6 键
  乙酸苯酯: 10 原子, 10 键
  合并后节点特征: torch.Size([20, 31])
  合并后边特征: torch.Size([38, 9])
  节点所属图: [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]


[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:38] DEPRECATIO

---

## 2. 图神经网络 (GNN) 编码器

GLN 提供了多种 GNN 变体，其中最基础的是 **Mean-Field GNN**（均场图神经网络）。

### 消息传递框架

所有 GNN 都遵循 **消息传递 (Message Passing)** 范式：

$$h_v^{(l+1)} = \sigma\left( W^{(l)} \cdot \text{AGG}\left(\{h_u^{(l)} : u \in \mathcal{N}(v)\}\right) + h_v^{(0)} \right)$$

其中：
- $h_v^{(l)}$：第 $l$ 层节点 $v$ 的嵌入
- $\mathcal{N}(v)$：节点 $v$ 的邻居
- $\text{AGG}$：聚合函数（sum/mean/max）
- $W^{(l)}$：第 $l$ 层的可学习参数
- $\sigma$：激活函数（GLN 默认用 tanh）

### 图级别读出 (Readout)

将所有节点嵌入聚合为一个图级别嵌入（即分子表示）：

$$h_G = \text{READOUT}\left(\{h_v^{(L)} : v \in G\}\right)$$

GLN 支持多种 readout 方式：`last`（只取最后一层）、`sum`、`mean`、`gru`。

### GLN 源码对应
- `gln/mods/mol_gnn/gnn_family/mean_field.py` → `EmbedMeanField`
- `gln/mods/mol_gnn/gnn_family/utils.py` → `GNNEmbedding`, `ReadoutNet`, `prepare_gnn`
- `gln/graph_logic/__init__.py` → `get_gnn()` (工厂函数)

In [4]:
# ====== 2.1 MLP 工具类 ======
# 对应 GLN 源码: gln/mods/mol_gnn/torch_util.py → MLP

NONLINEARITIES = {
    "tanh": nn.Tanh(),
    "relu": nn.ReLU(),
    "elu": nn.ELU(),
    "sigmoid": nn.Sigmoid(),
    "identity": nn.Identity(),
}


class MLP(nn.Module):
    """
    多层感知机。
    
    对应 GLN 源码: gln/mods/mol_gnn/torch_util.py → MLP
    """
    def __init__(self, input_dim, hidden_dims, nonlinearity='elu', act_last=None, bn=False):
        super(MLP, self).__init__()
        if isinstance(hidden_dims, str):
            hidden_dims = list(map(int, hidden_dims.split("-")))
        
        hidden_dims = [input_dim] + hidden_dims
        self.output_size = hidden_dims[-1]
        
        layers = []
        for i in range(1, len(hidden_dims)):
            layers.append(nn.Linear(hidden_dims[i - 1], hidden_dims[i]))
            if i + 1 < len(hidden_dims):  # 非最后一层
                if bn:
                    layers.append(nn.BatchNorm1d(hidden_dims[i]))
                layers.append(NONLINEARITIES[nonlinearity])
            else:
                if act_last is not None:
                    layers.append(NONLINEARITIES[act_last])
        
        self.main = nn.Sequential(*layers)
    
    def forward(self, z):
        return self.main(z)


print(f'MLP 工具类定义完成')

MLP 工具类定义完成


In [5]:
# ====== 2.2 ReadoutNet: 图级别读出网络 ======
# 对应 GLN 源码: gln/mods/mol_gnn/gnn_family/utils.py → ReadoutNet

class ReadoutNet(nn.Module):
    """
    从节点嵌入聚合为图级别嵌入。
    
    支持的读出方式:
    - 'last': 只取最后一层节点嵌入做聚合
    - 'sum'/'mean': 对所有层的节点嵌入求和/均值
    - 'gru': 用 GRU 逐层融合
    
    对应 GLN 源码: gln/mods/mol_gnn/gnn_family/utils.py → ReadoutNet
    """
    def __init__(self, node_state_dim, output_dim, max_lv, act_func='tanh',
                 out_method='last', readout_agg='sum', act_last=True, bn=False):
        super(ReadoutNet, self).__init__()
        self.out_method = out_method
        self.max_lv = max_lv
        self.act_last = act_last
        self.act_func = NONLINEARITIES[act_func]
        self.bn = bn
        
        # 聚合函数
        if readout_agg == 'sum':
            self.readout_agg = scatter_add
        elif readout_agg == 'mean':
            self.readout_agg = scatter_mean
        
        self.embed_dim = output_dim if output_dim > 0 else node_state_dim
        
        # 每层的读出投影
        if out_method == 'last':
            self.readout_funcs = nn.ModuleList([nn.Linear(node_state_dim, output_dim)])
        else:
            self.readout_funcs = nn.ModuleList(
                [nn.Linear(node_state_dim, output_dim) for _ in range(max_lv + 1)]
            )
        
        if out_method == 'gru':
            self.final_cell = nn.GRUCell(self.embed_dim, self.embed_dim)
    
    def forward(self, list_node_states, g_idx, num_graphs):
        """
        参数:
            list_node_states: [h^0, h^1, ..., h^L] 每层的节点嵌入
            g_idx: 每个节点所属的图索引
            num_graphs: 图的数量
        返回:
            graph_embed: [num_graphs, embed_dim] 图级别嵌入
        """
        if self.out_method == 'last':
            # 只取最后一层
            out = self.readout_funcs[0](list_node_states[-1])
            if self.act_last:
                out = self.act_func(out)
            graph_embed = self.readout_agg(out, g_idx.view(-1, 1).expand(-1, out.shape[1]),
                                           dim=0, dim_size=num_graphs)
            return graph_embed, (g_idx, out)
        
        # 对所有层做读出
        list_graph_embed = []
        for i in range(self.max_lv + 1):
            e = self.readout_funcs[i](list_node_states[i])
            if self.act_last:
                e = self.act_func(e)
            ge = self.readout_agg(e, g_idx.view(-1, 1).expand(-1, e.shape[1]),
                                  dim=0, dim_size=num_graphs)
            list_graph_embed.append(ge)
        
        out_embed = list_graph_embed[0]
        for i in range(1, self.max_lv + 1):
            if self.out_method == 'gru':
                out_embed = self.final_cell(list_graph_embed[i], out_embed)
            else:  # sum / mean
                out_embed = out_embed + list_graph_embed[i]
        
        if self.out_method == 'mean':
            out_embed = out_embed / (self.max_lv + 1)
        
        return out_embed, (None, None)


print('ReadoutNet 定义完成')

ReadoutNet 定义完成


In [6]:
# ====== 2.3 EmbedMeanField: 均场图神经网络 ======
# 对应 GLN 源码: gln/mods/mol_gnn/gnn_family/mean_field.py

from torch_geometric.nn.conv import MessagePassing


class _MeanFieldLayer(MessagePassing):
    """
    单层均场消息传递。
    
    操作: 对邻居节点特征求和 → 线性变换
    
    对应 GLN 源码: mean_field.py → _MeanFieldLayer
    """
    def __init__(self, latent_dim):
        super(_MeanFieldLayer, self).__init__(aggr='add')
        self.conv_params = nn.Linear(latent_dim, latent_dim)
    
    def forward(self, x, edge_index):
        x = x.unsqueeze(-1) if x.dim() == 1 else x
        return self.propagate(edge_index, x=x)
    
    def update(self, aggr_out):
        return self.conv_params(aggr_out)


class EmbedMeanField(nn.Module):
    """
    Mean-Field GNN 编码器。
    
    这是 GLN 默认使用的 GNN 架构。
    
    计算流程:
    1. 节点投影:  h^0 = σ(W_n · x_node + W_e · AGG(x_edge))
    2. 消息传递:  h^{l+1} = σ(W^l · AGG({h^l_u : u∈N(v)}) + h^0)
    3. 读出:      h_G = ReadoutNet(h^0, h^1, ..., h^L)
    
    核心特点:
    - 使用残差连接（加上初始输入 h^0）
    - 边特征通过 scatter_add 聚合到目标节点
    - 可共享或独立层参数
    
    对应 GLN 源码: gln/mods/mol_gnn/gnn_family/mean_field.py → EmbedMeanField
    """
    def __init__(self, latent_dim, output_dim, num_node_feats, num_edge_feats,
                 max_lv=3, act_func='tanh', readout_agg='sum', act_last=True):
        super(EmbedMeanField, self).__init__()
        self.latent_dim = latent_dim
        self.output_dim = output_dim
        self.max_lv = max_lv
        self.act_func = NONLINEARITIES[act_func]
        self.embed_dim = output_dim if output_dim > 0 else latent_dim
        
        # 初始投影层
        self.w_n2l = nn.Linear(num_node_feats, latent_dim)  # 节点特征 → 隐藏维度
        if num_edge_feats > 0:
            self.w_e2l = nn.Linear(num_edge_feats, latent_dim)  # 边特征 → 隐藏维度
        self.num_edge_feats = num_edge_feats
        
        # 消息传递层（共享参数）
        self.conv_layer = _MeanFieldLayer(latent_dim)
        
        # 读出网络
        self.readout_net = ReadoutNet(
            node_state_dim=latent_dim,
            output_dim=output_dim,
            max_lv=max_lv,
            act_func=act_func,
            out_method='last',
            readout_agg=readout_agg,
            act_last=act_last
        )
    
    def forward(self, graph_list):
        """
        参数:
            graph_list: MolGraph 对象列表
        返回:
            graph_embed: [num_graphs, embed_dim] 图级别嵌入
            nodes_info: (g_idx, node_embed) 节点级别信息
        """
        # ---- Step 1: 准备批次数据 ----
        # 过滤 None 并记录位置
        selected = []
        sublist = []
        for i, g in enumerate(graph_list):
            if g is not None:
                selected.append(i)
                sublist.append(g)
        
        if not sublist:
            return torch.zeros(len(graph_list), self.embed_dim).to(DEVICE), None
        
        node_feat, edge_feat, edge_from, edge_to, g_idx = prepare_batch(sublist)
        
        # ---- Step 2: 初始节点嵌入 ----
        input_node_linear = self.w_n2l(node_feat)
        input_message = input_node_linear
        
        if edge_feat is not None and self.num_edge_feats > 0:
            input_edge_linear = self.w_e2l(edge_feat)
            # 将边特征聚合到目标节点
            e2n = scatter_add(input_edge_linear, edge_to, dim=0, dim_size=node_feat.shape[0])
            input_message = input_message + e2n
        
        input_potential = self.act_func(input_message)  # h^0
        
        # ---- Step 3: 消息传递 (L 层) ----
        edge_index = torch.stack([edge_from, edge_to])
        cur_state = input_potential
        all_embeds = [cur_state]
        
        for lv in range(self.max_lv):
            node_linear = self.conv_layer(cur_state, edge_index)
            merged = node_linear + input_message  # 残差连接
            cur_state = self.act_func(merged)
            all_embeds.append(cur_state)
        
        # ---- Step 4: 读出 ----
        embed, nodes_info = self.readout_net(all_embeds, g_idx, len(sublist))
        
        # 处理 None 图的情况
        if len(sublist) != len(graph_list):
            full_embed = torch.zeros(len(graph_list), self.embed_dim).to(DEVICE)
            full_embed[selected] = embed
            return full_embed, None
        
        return embed, nodes_info


# ====== 演示 GNN 编码 ======
gnn = EmbedMeanField(
    latent_dim=64,
    output_dim=32,
    num_node_feats=NUM_NODE_FEATS,
    num_edge_feats=NUM_EDGE_FEATS,
    max_lv=3,
    act_func='tanh',
    readout_agg='sum'
).to(DEVICE)

# 编码三个分子
graphs = [
    SmilesMols.get_mol_graph('CC(=O)O'),       # 乙酸
    SmilesMols.get_mol_graph('c1ccccc1'),       # 苯
    SmilesMols.get_mol_graph('CC(=O)Oc1ccccc1') # 乙酸苯酯
]

with torch.no_grad():
    embed, _ = gnn(graphs)

print(f'GNN 编码结果:')
print(f'  配置: latent_dim=64, output_dim=32, max_lv=3, act=tanh')
print(f'  参数量: {sum(p.numel() for p in gnn.parameters()):,}')
print(f'  输入: 3 个分子图')
print(f'  输出形状: {embed.shape}  (3个分子 × 32维嵌入)')
print(f'  乙酸嵌入 (前8维): {embed[0, :8].tolist()}')
print(f'  苯嵌入 (前8维):   {embed[1, :8].tolist()}')

GNN 编码结果:
  配置: latent_dim=64, output_dim=32, max_lv=3, act=tanh
  参数量: 8,928
  输入: 3 个分子图
  输出形状: torch.Size([3, 32])  (3个分子 × 32维嵌入)
  乙酸嵌入 (前8维): [1.8586729764938354, -0.08870276063680649, 0.6024036407470703, -1.6639976501464844, 2.3112497329711914, 2.2583112716674805, 0.7162607908248901, -0.0023190975189208984]
  苯嵌入 (前8维):   [0.38324612379074097, 1.0819504261016846, -0.762947678565979, 0.8511421084403992, 2.372440814971924, 0.9243211150169373, -1.7724573612213135, -1.0350078344345093]


---

## 3. Jagged Softmax: 变长序列的归一化

### 为什么需要 Jagged Softmax？

在 GLN 的推理过程中，不同产物的候选中心/模板/反应物数量是**不同**的。例如：
- 产物A有 5 个候选中心
- 产物B有 12 个候选中心
- 产物C有 3 个候选中心

标准 softmax 需要每个样本的候选列表**等长**。而 GLN 使用 **Jagged (锯齿形) Softmax**：将所有候选拼成一个长向量，但只在每个样本内部做 softmax 归一化。

```
                样本1         样本2      样本3
候选:         [a, b, c]    [d, e]     [f, g, h, i]
拼接为:       [a, b, c,     d, e,      f, g, h, i]
prefix_sum:   [     3,        5,              9   ]
softmax范围:  [──────]     [────]     [──────────]
```

GLN 原始代码通过 C++ 和 CUDA 扩展实现高效 Jagged Softmax。这里我们用纯 PyTorch 实现。

### GLN 源码对应
- `gln/mods/torchext/jagged_ops.py` → `JaggedLogSoftmax`
- `gln/mods/torchext/src/extlib.cpp` → C++ 实现
- `gln/mods/torchext/src/extlib_cuda.cu` → CUDA 实现

In [7]:
# ====== 3. Jagged Log Softmax 实现 ======
# 对应 GLN 源码: gln/mods/torchext/jagged_ops.py
# 原始代码使用 C++/CUDA 实现高性能版本，这里用纯 PyTorch 等价实现

def jagged_log_softmax(logits, prefix_sum):
    """
    对变长序列的 logits 做分段 log_softmax。
    
    参数:
        logits: [N] 一维张量，所有候选的 logit 值
        prefix_sum: [B] 一维张量，每个样本的候选结束位置
    
    返回:
        log_prob: [N] 每个候选的 log 概率（在其所属样本内归一化）
    
    示例:
        logits = [1.0, 2.0, 3.0, 0.5, 1.5]  (样本0有3个候选，样本1有2个)
        prefix_sum = [3, 5]
        输出: 对 [1,2,3] 和 [0.5,1.5] 分别做 log_softmax
    """
    log_prob = torch.zeros_like(logits)
    
    start = 0
    for end in prefix_sum:
        end = end.item()
        segment = logits[start:end]
        log_prob[start:end] = F.log_softmax(segment, dim=0)
        start = end
    
    return log_prob


# ====== 演示 ======
# 假设3个样本，候选数分别为 3, 2, 4
logits = torch.tensor([1.0, 2.0, 3.0,     # 样本0: 3个候选
                        0.5, 1.5,           # 样本1: 2个候选
                        -1.0, 0.0, 1.0, 2.0 # 样本2: 4个候选
                       ], requires_grad=True)
prefix_sum = torch.LongTensor([3, 5, 9])

log_probs = jagged_log_softmax(logits, prefix_sum)

print('Jagged Log Softmax 演示:')
print('=' * 60)
print(f'输入 logits: {logits.tolist()}')
print(f'prefix_sum:  {prefix_sum.tolist()}')
print(f'输出 log概率: {[f"{v:.4f}" for v in log_probs.tolist()]}')
print()
print('分段验证:')
print(f'  样本0 (候选3个): {[f"{v:.4f}" for v in log_probs[:3].tolist()]}')
print(f'    → 概率: {[f"{np.exp(v):.4f}" for v in log_probs[:3].tolist()]}')
print(f'    → 概率和: {sum(np.exp(v) for v in log_probs[:3].tolist()):.4f} (应=1.0)')
print(f'  样本1 (候选2个): {[f"{v:.4f}" for v in log_probs[3:5].tolist()]}')
print(f'    → 概率: {[f"{np.exp(v):.4f}" for v in log_probs[3:5].tolist()]}')
print(f'    → 概率和: {sum(np.exp(v) for v in log_probs[3:5].tolist()):.4f} (应=1.0)')

# 验证梯度可传播
loss = -log_probs[2]  # 样本0的正确答案是第3个候选
loss.backward()
print(f'\n梯度传播验证: logits.grad = {logits.grad.tolist()}')

Jagged Log Softmax 演示:
输入 logits: [1.0, 2.0, 3.0, 0.5, 1.5, -1.0, 0.0, 1.0, 2.0]
prefix_sum:  [3, 5, 9]
输出 log概率: ['-2.4076', '-1.4076', '-0.4076', '-1.3133', '-0.3133', '-3.4402', '-2.4402', '-1.4402', '-0.4402']

分段验证:
  样本0 (候选3个): ['-2.4076', '-1.4076', '-0.4076']
    → 概率: ['0.0900', '0.2447', '0.6652']
    → 概率和: 1.0000 (应=1.0)
  样本1 (候选2个): ['-1.3133', '-0.3133']
    → 概率: ['0.2689', '0.7311']
    → 概率和: 1.0000 (应=1.0)

梯度传播验证: logits.grad = [0.09003058075904846, 0.24472849071025848, -0.334758996963501, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


---

## 4. 三个概率计算模块

GLN 的核心模型由三个 **概率计算模块 (Predicate)** 组成，形成层次化推理：

```
产物 p
 ├─→ CenterProbCalc:  P(c|p)      选择反应中心
 │    GNN编码产物 × GNN/Onehot编码中心 → 注意力评分 → Jagged Softmax
 │
 ├─→ ActiveProbCalc:  P(t|c,p)    选择模板
 │    GNN编码产物 × DeepSet/Onehot编码模板 → 注意力评分 → Jagged Softmax
 │
 └─→ ReactionProbCalc: P(r|t,p)   验证反应物
      GNN编码产物 × DeepSet编码反应物 → 注意力评分 → Jagged Softmax
```

### 注意力评分机制

三个模块都使用注意力机制来计算产物与候选项之间的兼容性分数：

| 类型 | 公式 | 说明 |
|------|------|------|
| inner_prod | $s = \mathbf{p}^\top \mathbf{c}$ | 内积，最简单 |
| mlp | $s = \text{MLP}([\mathbf{p}; \mathbf{c}])$ | MLP 拼接 |
| bilinear | $s = \mathbf{p}^\top W \mathbf{c}$ | 双线性 |

### GLN 源码对应
- `gln/graph_logic/soft_logic.py` → `CenterProbCalc`, `ActiveProbCalc`, `ReactionProbCalc`
- `gln/graph_logic/soft_logic.py` → `jagged_forward()` (通用前向计算)

In [8]:
# ====== 4.1 通用前向计算: jagged_forward ======
# 对应 GLN 源码: gln/graph_logic/soft_logic.py → jagged_forward()
# 这是三个预测模块共享的核心计算逻辑

def jagged_forward(list_graph, list_of_list_cand, graph_enc, cand_enc,
                   att_func, list_target_pos=None, normalize=True):
    """
    Jagged (变长) 前向计算：
    1. GNN 编码产物分子 → graph_embed
    2. 编码所有候选项 → cand_embed  
    3. 扩展产物嵌入以匹配候选数
    4. 计算注意力分数
    5. Jagged Softmax 归一化（在每个样本内部）
    6. 返回正确目标的 log 概率
    
    对应 GLN 源码: soft_logic.py → jagged_forward()
    
    参数:
        list_graph: 产物分子图列表
        list_of_list_cand: 每个产物的候选列表
        graph_enc: 产物编码器 (GNN)
        cand_enc: 候选编码器 (GNN/Onehot/Deepset)
        att_func: 注意力评分函数
        list_target_pos: 每个样本中正确答案的位置索引
        normalize: 是否做 softmax 归一化
    """
    # Step 1: 编码产物
    graph_embed = graph_enc(list_graph)
    
    # Step 2: 将所有候选展平为一个列表
    flat_cands = []
    rep_indices = []  # 每个候选对应哪个产物
    prefix_sum = []
    offset = 0
    for i, cands in enumerate(list_of_list_cand):
        for c in cands:
            flat_cands.append(c)
        rep_indices += [i] * len(cands)
        offset += len(cands)
        prefix_sum.append(offset)
    
    # Step 3: 编码候选
    cand_embed = cand_enc(flat_cands)
    rep_indices = torch.LongTensor(rep_indices).to(DEVICE)
    prefix_sum = torch.LongTensor(prefix_sum).to(DEVICE)
    
    # Step 4: 扩展产物嵌入（复制到对应的候选位置）
    graph_embed_expanded = torch.index_select(graph_embed, 0, rep_indices)
    
    # Step 5: 注意力评分
    logits = att_func(graph_embed_expanded, cand_embed)
    
    # Step 6: 归一化
    if normalize:
        log_prob = jagged_log_softmax(logits, prefix_sum)
    else:
        log_prob = logits
    
    # Step 7: 提取目标位置的 log 概率
    if list_target_pos is None:
        return log_prob
    
    target_pos = []
    offset = 0
    for i, cands in enumerate(list_of_list_cand):
        idx = list_target_pos[i]
        target_pos.append(offset + idx)
        offset += len(cands)
    target_pos = torch.LongTensor(target_pos).to(DEVICE)
    
    return log_prob[target_pos]


print('jagged_forward 通用前向计算定义完成')
print()
print('计算流程示意:')
print('''
  产物列表 [p1, p2]
  候选列表 [[c1a, c1b, c1c], [c2a, c2b]]  (p1有3候选, p2有2候选)
    │
    ├── graph_enc([p1, p2]) → [embed_p1, embed_p2]
    ├── cand_enc([c1a, c1b, c1c, c2a, c2b]) → [embed_c1a, ...]
    │
    ├── 扩展产物嵌入: [embed_p1, embed_p1, embed_p1, embed_p2, embed_p2]
    ├── 注意力评分: att(product_expanded, candidates) → [s1a, s1b, s1c, s2a, s2b]
    │
    └── Jagged Softmax:
         样本1: softmax([s1a, s1b, s1c]) → [log_p1a, log_p1b, log_p1c]
         样本2: softmax([s2a, s2b])       → [log_p2a, log_p2b]
''')

jagged_forward 通用前向计算定义完成

计算流程示意:

  产物列表 [p1, p2]
  候选列表 [[c1a, c1b, c1c], [c2a, c2b]]  (p1有3候选, p2有2候选)
    │
    ├── graph_enc([p1, p2]) → [embed_p1, embed_p2]
    ├── cand_enc([c1a, c1b, c1c, c2a, c2b]) → [embed_c1a, ...]
    │
    ├── 扩展产物嵌入: [embed_p1, embed_p1, embed_p1, embed_p2, embed_p2]
    ├── 注意力评分: att(product_expanded, candidates) → [s1a, s1b, s1c, s2a, s2b]
    │
    └── Jagged Softmax:
         样本1: softmax([s1a, s1b, s1c]) → [log_p1a, log_p1b, log_p1c]
         样本2: softmax([s2a, s2b])       → [log_p2a, log_p2b]



In [9]:
# ====== 4.2 CenterProbCalc: 反应中心预测 ======
# 对应 GLN 源码: gln/graph_logic/soft_logic.py → CenterProbCalc

class OnehotEmbedder(nn.Module):
    """
    One-hot 嵌入编码器。
    为每个已知的 key 分配一个可学习的嵌入向量。
    
    对应 GLN 源码: soft_logic.py → OnehotEmbedder
    
    用途:
    - CenterProbCalc 中编码产物中心 (当 subg_enc='onehot')
    - ActiveProbCalc 中编码模板 (当 tpl_enc='onehot')
    """
    def __init__(self, list_keys, fn_getkey, embed_size):
        super(OnehotEmbedder, self).__init__()
        self.key_idx = {key: i for i, key in enumerate(list_keys)}
        self.fn_getkey = fn_getkey
        self.embedding = nn.Embedding(len(list_keys) + 1, embed_size)  # +1 for unknown
    
    def forward(self, list_objs):
        indices = []
        for obj in list_objs:
            key = self.fn_getkey(obj)
            indices.append(self.key_idx.get(key, len(self.key_idx)))
        indices = torch.LongTensor(indices).to(DEVICE)
        return self.embedding(indices)


class CenterProbCalc(nn.Module):
    """
    反应中心预测模块。
    
    给定产物 p 和候选中心集合 {c1, c2, ...}:
    1. GNN 编码产物 → embed_p
    2. GNN/Onehot 编码每个中心 → embed_ci
    3. 计算注意力分数: score_i = att(embed_p, embed_ci)
    4. Jagged Softmax 归一化
    
    对应 GLN 源码: soft_logic.py → CenterProbCalc
    """
    def __init__(self, embed_dim, prod_cano_smarts=None, subg_enc='onehot', att_type='inner_prod'):
        super(CenterProbCalc, self).__init__()
        
        # 产物编码器
        self.prod_enc = EmbedMeanField(
            latent_dim=embed_dim, output_dim=embed_dim,
            num_node_feats=NUM_NODE_FEATS, num_edge_feats=NUM_EDGE_FEATS,
            max_lv=3, act_func='tanh'
        )
        
        # 中心编码器
        if subg_enc == 'onehot' and prod_cano_smarts is not None:
            self.center_enc = OnehotEmbedder(
                list_keys=prod_cano_smarts,
                fn_getkey=lambda m: m.name if m is not None else None,
                embed_size=embed_dim
            )
            self.center_embed_func = lambda x: self.center_enc(x)
        else:
            # 用 GNN 编码 SMARTS 子结构
            self.center_enc = EmbedMeanField(
                latent_dim=embed_dim, output_dim=embed_dim,
                num_node_feats=NUM_NODE_FEATS, num_edge_feats=NUM_EDGE_FEATS,
                max_lv=3, act_func='tanh'
            )
            self.center_embed_func = lambda x: self.center_enc(x)[0]
        
        # 注意力函数
        if att_type == 'inner_prod':
            self.att_func = lambda x, y: torch.sum(x * y, dim=1).view(-1)
        elif att_type == 'mlp':
            self.pred = MLP(2 * embed_dim, [embed_dim, 1], nonlinearity='relu')
            self.att_func = lambda x, y: self.pred(torch.cat((x, y), dim=1)).view(-1)
        elif att_type == 'bilinear':
            self.bilin = nn.Bilinear(embed_dim, embed_dim, 1)
            self.att_func = lambda x, y: self.bilin(x, y).view(-1)
    
    def forward(self, list_mols, list_of_list_centers, list_target_pos=None):
        if list_target_pos is None:
            list_target_pos = [0] * len(list_mols)
        return jagged_forward(
            list_mols, list_of_list_centers,
            graph_enc=lambda x: self.prod_enc(x)[0],
            cand_enc=self.center_embed_func,
            att_func=self.att_func,
            list_target_pos=list_target_pos
        )
    
    def inference(self, list_mols, list_of_list_centers):
        """推理模式：返回 logits（不归一化）"""
        return jagged_forward(
            list_mols, list_of_list_centers,
            graph_enc=lambda x: self.prod_enc(x)[0],
            cand_enc=self.center_embed_func,
            att_func=self.att_func,
            list_target_pos=None,
            normalize=False
        )


print('CenterProbCalc 定义完成')
print()
print('CenterProbCalc 架构:')
print('''
  输入: 产物分子 p, 候选中心列表 [c1, c2, ...]
    │
    ├── prod_enc(p):       GNN → embed_p ∈ R^d
    ├── center_enc([c_i]): GNN/Onehot → embed_c_i ∈ R^d
    │
    ├── att_func(embed_p, embed_c_i) → score_i
    │
    └── jagged_log_softmax([s1, s2, ...]) → P(c_i | p)
''')

CenterProbCalc 定义完成

CenterProbCalc 架构:

  输入: 产物分子 p, 候选中心列表 [c1, c2, ...]
    │
    ├── prod_enc(p):       GNN → embed_p ∈ R^d
    ├── center_enc([c_i]): GNN/Onehot → embed_c_i ∈ R^d
    │
    ├── att_func(embed_p, embed_c_i) → score_i
    │
    └── jagged_log_softmax([s1, s2, ...]) → P(c_i | p)



In [10]:
# ====== 4.3 DeepsetTempFeaturizer: 模板特征编码器 ======
# 对应 GLN 源码: gln/graph_logic/graph_feat.py → DeepsetTempFeaturizer

class DeepsetTempFeaturizer(nn.Module):
    """
    用 Deepset 方式编码反应模板。
    
    模板格式: [产物中心SMARTS]>[离去基团]>[反应物中心SMARTS]
    
    编码方式:
    1. GNN 编码产物中心 SMARTS → prod_embed
    2. GNN 编码每个反应物中心 SMARTS → react_embeds
    3. Sum 聚合所有反应物嵌入 → aggregated_react
    4. 拼接: concat(prod_embed, aggregated_react)
    5. MLP 投影到最终嵌入
    
    对应 GLN 源码: graph_feat.py → DeepsetTempFeaturizer
    """
    def __init__(self, embed_dim):
        super(DeepsetTempFeaturizer, self).__init__()
        self.prod_gnn = EmbedMeanField(
            latent_dim=embed_dim, output_dim=embed_dim,
            num_node_feats=NUM_NODE_FEATS, num_edge_feats=NUM_EDGE_FEATS,
            max_lv=3, act_func='tanh'
        )
        self.react_gnn = EmbedMeanField(
            latent_dim=embed_dim, output_dim=embed_dim,
            num_node_feats=NUM_NODE_FEATS, num_edge_feats=NUM_EDGE_FEATS,
            max_lv=3, act_func='tanh'
        )
        self.readout = MLP(embed_dim * 2, [embed_dim, embed_dim], nonlinearity='relu')
    
    def forward(self, template_list):
        """
        参数:
            template_list: 模板字符串列表 ["prod>agent>react", ...]
        """
        list_prod = []
        list_reactants = []
        reactant_indices = []
        
        for i, temp in enumerate(template_list):
            prod, _, reacts = temp.split('>')
            list_prod.append(SmartsMols.get_mol_graph(prod))
            for r in reacts.split('.'):
                list_reactants.append(SmartsMols.get_mol_graph(r))
                reactant_indices.append(i)
        
        # GNN 编码
        prods_embed, _ = self.prod_gnn(list_prod)
        reacts_embed, _ = self.react_gnn(list_reactants)
        reacts_embed = F.relu(reacts_embed)
        
        # Sum 聚合各模板的反应物
        reactant_indices = torch.LongTensor(reactant_indices).to(DEVICE)
        agg_reacts = scatter_add(
            reacts_embed,
            reactant_indices.view(-1, 1).expand(-1, reacts_embed.shape[1]),
            dim=0, dim_size=len(template_list)
        )
        
        # 拼接 + MLP
        feats = torch.cat((prods_embed, agg_reacts), dim=1)
        return self.readout(feats)


class DeepsetReactFeaturizer(nn.Module):
    """
    用 Deepset 方式编码反应物。
    
    编码方式:
    1. GNN 编码每个反应物分子 → react_embeds
    2. Sum 聚合同一反应的所有反应物 → aggregated
    
    对应 GLN 源码: graph_feat.py → DeepsetReactFeaturizer
    """
    def __init__(self, embed_dim):
        super(DeepsetReactFeaturizer, self).__init__()
        self.react_gnn = EmbedMeanField(
            latent_dim=embed_dim, output_dim=embed_dim,
            num_node_feats=NUM_NODE_FEATS, num_edge_feats=NUM_EDGE_FEATS,
            max_lv=3, act_func='tanh'
        )
    
    def forward(self, reaction_list):
        """
        参数:
            reaction_list: 反应字符串列表 ["reactants>>product", ...]
        """
        list_reactants = []
        reactant_indices = []
        
        for i, react in enumerate(reaction_list):
            reactants, _, prod = react.split('>')
            for r in reactants.split('.'):
                list_reactants.append(SmilesMols.get_mol_graph(r))
                reactant_indices.append(i)
        
        reacts_embed, _ = self.react_gnn(list_reactants)
        
        reactant_indices = torch.LongTensor(reactant_indices).to(DEVICE)
        agg_reacts = scatter_add(
            reacts_embed,
            reactant_indices.view(-1, 1).expand(-1, reacts_embed.shape[1]),
            dim=0, dim_size=len(reaction_list)
        )
        return agg_reacts


print('DeepsetTempFeaturizer 和 DeepsetReactFeaturizer 定义完成')
print()
print('DeepsetTempFeaturizer 架构:')
print('''
  模板: "[C:1](=O)O>[离去基]>[C:1](=O)[OH].[OH]"
    │
    ├── prod_gnn("[C:1](=O)O") → prod_embed ∈ R^d
    ├── react_gnn("[C:1](=O)[OH]") → r1_embed ∈ R^d
    ├── react_gnn("[OH]")          → r2_embed ∈ R^d
    │
    ├── Sum聚合: agg_react = r1_embed + r2_embed
    ├── 拼接: feat = [prod_embed; agg_react] ∈ R^{2d}
    │
    └── MLP(feat) → template_embed ∈ R^d
''')

DeepsetTempFeaturizer 和 DeepsetReactFeaturizer 定义完成

DeepsetTempFeaturizer 架构:

  模板: "[C:1](=O)O>[离去基]>[C:1](=O)[OH].[OH]"
    │
    ├── prod_gnn("[C:1](=O)O") → prod_embed ∈ R^d
    ├── react_gnn("[C:1](=O)[OH]") → r1_embed ∈ R^d
    ├── react_gnn("[OH]")          → r2_embed ∈ R^d
    │
    ├── Sum聚合: agg_react = r1_embed + r2_embed
    ├── 拼接: feat = [prod_embed; agg_react] ∈ R^{2d}
    │
    └── MLP(feat) → template_embed ∈ R^d



In [11]:
# ====== 4.4 ActiveProbCalc: 模板预测模块 ======
# 对应 GLN 源码: gln/graph_logic/soft_logic.py → ActiveProbCalc

class ActiveProbCalc(nn.Module):
    """
    模板预测模块。
    
    给定产物 p 和候选模板集合 {t1, t2, ...}:
    1. GNN 编码产物 → embed_p
    2. DeepSet/Onehot 编码每个模板 → embed_ti
    3. 注意力评分 → Jagged Softmax
    
    对应 GLN 源码: soft_logic.py → ActiveProbCalc
    """
    def __init__(self, embed_dim, tpl_enc_type='deepset', unique_templates=None, att_type='inner_prod'):
        super(ActiveProbCalc, self).__init__()
        self.prod_enc = EmbedMeanField(
            latent_dim=embed_dim, output_dim=embed_dim,
            num_node_feats=NUM_NODE_FEATS, num_edge_feats=NUM_EDGE_FEATS,
            max_lv=3, act_func='tanh'
        )
        
        if tpl_enc_type == 'deepset':
            self.tpl_enc = DeepsetTempFeaturizer(embed_dim)
        elif tpl_enc_type == 'onehot' and unique_templates is not None:
            self.tpl_enc = OnehotEmbedder(
                list_keys=unique_templates,
                fn_getkey=lambda x: x,
                embed_size=embed_dim
            )
        
        if att_type == 'inner_prod':
            self.att_func = lambda x, y: torch.sum(x * y, dim=1).view(-1)
        elif att_type == 'mlp':
            self.pred = MLP(2 * embed_dim, [embed_dim, 1], nonlinearity='relu')
            self.att_func = lambda x, y: self.pred(torch.cat((x, y), dim=1)).view(-1)
        elif att_type == 'bilinear':
            self.bilin = nn.Bilinear(embed_dim, embed_dim, 1)
            self.att_func = lambda x, y: self.bilin(x, y).view(-1)
    
    def forward(self, list_mols, list_of_list_templates, list_target_pos=None):
        if list_target_pos is None:
            list_target_pos = [0] * len(list_mols)
        return jagged_forward(
            list_mols, list_of_list_templates,
            graph_enc=lambda x: self.prod_enc(x)[0],
            cand_enc=lambda x: self.tpl_enc(x),
            att_func=self.att_func,
            list_target_pos=list_target_pos
        )
    
    def inference(self, list_mols, list_of_list_templates):
        return jagged_forward(
            list_mols, list_of_list_templates,
            graph_enc=lambda x: self.prod_enc(x)[0],
            cand_enc=lambda x: self.tpl_enc(x),
            att_func=self.att_func,
            list_target_pos=None,
            normalize=False
        )


# ====== 4.5 ReactionProbCalc: 反应物验证模块 ======
# 对应 GLN 源码: gln/graph_logic/soft_logic.py → ReactionProbCalc

class ReactionProbCalc(nn.Module):
    """
    反应物验证模块。
    
    给定产物 p 和候选反应 {r1>>p, r2>>p, ...}:
    1. GNN 编码产物 → embed_p
    2. DeepSet 编码每个候选反应物集 → embed_ri
    3. 注意力评分 → Jagged Softmax
    
    对应 GLN 源码: soft_logic.py → ReactionProbCalc
    """
    def __init__(self, embed_dim, att_type='inner_prod'):
        super(ReactionProbCalc, self).__init__()
        self.prod_enc = EmbedMeanField(
            latent_dim=embed_dim, output_dim=embed_dim,
            num_node_feats=NUM_NODE_FEATS, num_edge_feats=NUM_EDGE_FEATS,
            max_lv=3, act_func='tanh'
        )
        self.react_enc = DeepsetReactFeaturizer(embed_dim)
        
        if att_type == 'inner_prod':
            self.att_func = lambda x, y: torch.sum(x * y, dim=1).view(-1)
        elif att_type == 'mlp':
            self.pred = MLP(2 * embed_dim, [embed_dim, 1], nonlinearity='relu')
            self.att_func = lambda x, y: self.pred(torch.cat((x, y), dim=1)).view(-1)
        elif att_type == 'bilinear':
            self.bilin = nn.Bilinear(embed_dim, embed_dim, 1)
            self.att_func = lambda x, y: self.bilin(x, y).view(-1)
    
    def forward(self, list_mols, list_of_list_reactions, list_target_pos=None):
        if list_target_pos is None:
            list_target_pos = [0] * len(list_mols)
        return jagged_forward(
            list_mols, list_of_list_reactions,
            graph_enc=lambda x: self.prod_enc(x)[0],
            cand_enc=lambda x: self.react_enc(x),
            att_func=self.att_func,
            list_target_pos=list_target_pos
        )
    
    def inference(self, list_mols, list_of_list_reactions):
        return jagged_forward(
            list_mols, list_of_list_reactions,
            graph_enc=lambda x: self.prod_enc(x)[0],
            cand_enc=lambda x: self.react_enc(x),
            att_func=self.att_func,
            list_target_pos=None,
            normalize=False
        )


print('ActiveProbCalc 和 ReactionProbCalc 定义完成')

ActiveProbCalc 和 ReactionProbCalc 定义完成


---

## 5. 完整的 GraphPath 模型

`GraphPath` 是 GLN 的**顶层模型**，它将三个概率计算模块组合在一起，形成一个端到端的训练/推理系统。

### 训练时的损失函数

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^{B} \left[ \log P(c_i | p_i) + \log P(t_i | c_i, p_i) + \log P(r_i | t_i, p_i) \right]$$

其中 $B$ 是 batch size，每一项都通过 Jagged Softmax 在正负样本之间计算。

### GLN 源码对应
- `gln/graph_logic/logic_net.py` → `GraphPath`

In [12]:
# ====== 5.1 GraphPath: 完整 GLN 模型 ======
# 对应 GLN 源码: gln/graph_logic/logic_net.py → GraphPath

class DataSample:
    """
    训练数据样本。
    
    对应 GLN 源码: gln/training/data_gen.py → DataSample
    
    属性:
        prod: 产物的 canonical SMILES
        center: 正确的反应中心 SMARTS (正样本)
        template: 正确的完整模板 SMARTS (正样本)
        reaction: 正确的反应 "reactants>>product" (正样本)
        neg_centers: 负采样的中心列表
        neg_tpls: 负采样的模板列表
        neg_reactions: 负采样的反应列表
    """
    def __init__(self, prod, center, template, neg_centers=None, neg_tpls=None,
                 reaction=None, neg_reactions=None):
        self.prod = prod
        self.center = center
        self.template = template
        self.neg_centers = neg_centers or []
        self.neg_tpls = neg_tpls or []
        self.reaction = reaction
        self.neg_reactions = neg_reactions or []


class GraphPath(nn.Module):
    """
    GLN 的完整模型 (Graph Logic Network)。
    
    组合三个概率预测模块：
    1. prod_center_predicate (CenterProbCalc): 预测反应中心
    2. tpl_fwd_predicate (ActiveProbCalc): 预测模板  
    3. reaction_predicate (ReactionProbCalc): 预测反应物
    
    训练损失:
        L = -mean(log P(center|prod)) - mean(log P(template|prod)) - mean(log P(reactants|prod))
    
    对应 GLN 源码: gln/graph_logic/logic_net.py → GraphPath
    """
    def __init__(self, embed_dim=32, att_type='inner_prod', retro_during_train=True,
                 prod_cano_smarts=None, unique_templates=None, tpl_enc='deepset'):
        super(GraphPath, self).__init__()
        
        self.prod_center_predicate = CenterProbCalc(
            embed_dim=embed_dim,
            prod_cano_smarts=prod_cano_smarts,
            att_type=att_type
        )
        self.tpl_fwd_predicate = ActiveProbCalc(
            embed_dim=embed_dim,
            tpl_enc_type=tpl_enc,
            unique_templates=unique_templates,
            att_type=att_type
        )
        self.reaction_predicate = ReactionProbCalc(
            embed_dim=embed_dim,
            att_type=att_type
        )
        self.retro_during_train = retro_during_train
    
    def forward(self, samples):
        """
        训练前向传播。
        
        参数:
            samples: DataSample 对象列表
        返回:
            loss: 标量损失值
        """
        prods = []
        list_of_list_centers = []
        list_of_list_tpls = []
        list_of_list_reacts = []
        
        for sample in samples:
            prods.append(SmilesMols.get_mol_graph(sample.prod))
            
            # 中心列表: [正确中心, 负样本中心1, 负样本中心2, ...]
            centers = [SmartsMols.get_mol_graph(sample.center)]
            centers.extend([SmartsMols.get_mol_graph(c) for c in sample.neg_centers])
            list_of_list_centers.append(centers)
            
            # 模板列表: [正确模板, 负样本模板1, ...]
            tpls = [sample.template] + sample.neg_tpls
            list_of_list_tpls.append(tpls)
            
            # 反应列表 (可选)
            if self.retro_during_train and sample.reaction:
                reacts = [sample.reaction] + sample.neg_reactions
                list_of_list_reacts.append(reacts)
        
        # ---- 计算三项损失 ----
        # 中心预测损失 (正确中心在位置0)
        center_log_prob = self.prod_center_predicate(prods, list_of_list_centers)
        # 模板预测损失 (正确模板在位置0)
        tpl_log_prob = self.tpl_fwd_predicate(prods, list_of_list_tpls)
        
        loss = -torch.mean(center_log_prob) - torch.mean(tpl_log_prob)
        
        if self.retro_during_train and list_of_list_reacts:
            react_log_prob = self.reaction_predicate(prods, list_of_list_reacts)
            loss = loss - torch.mean(react_log_prob)
        
        return loss


print('GraphPath (完整 GLN 模型) 定义完成')
print()
print('GraphPath 架构总览:')
print('''
  ┌─────────────────────────────────────────────────────────┐
  │                    GraphPath                            │
  │                                                         │
  │  输入: DataSample (产物, 正中心, 正模板, 正反应物,     │
  │                     负中心列表, 负模板列表, 负反应物)   │
  │                                                         │
  │  ┌──────────────────────────┐                           │
  │  │ CenterProbCalc           │ → log P(center|prod)     │
  │  │  prod_enc(GNN) + center_enc(GNN/Onehot)             │
  │  │  + inner_prod attention  │                           │
  │  └──────────────────────────┘                           │
  │                                                         │
  │  ┌──────────────────────────┐                           │
  │  │ ActiveProbCalc           │ → log P(template|prod)   │
  │  │  prod_enc(GNN) + tpl_enc(DeepSet/Onehot)            │
  │  │  + inner_prod attention  │                           │
  │  └──────────────────────────┘                           │
  │                                                         │
  │  ┌──────────────────────────┐                           │
  │  │ ReactionProbCalc         │ → log P(react|prod)      │
  │  │  prod_enc(GNN) + react_enc(DeepSet)                  │
  │  │  + inner_prod attention  │                           │
  │  └──────────────────────────┘                           │
  │                                                         │
  │  Loss = -mean(log_p_center + log_p_tpl + log_p_react)  │
  └─────────────────────────────────────────────────────────┘
''')

GraphPath (完整 GLN 模型) 定义完成

GraphPath 架构总览:

  ┌─────────────────────────────────────────────────────────┐
  │                    GraphPath                            │
  │                                                         │
  │  输入: DataSample (产物, 正中心, 正模板, 正反应物,     │
  │                     负中心列表, 负模板列表, 负反应物)   │
  │                                                         │
  │  ┌──────────────────────────┐                           │
  │  │ CenterProbCalc           │ → log P(center|prod)     │
  │  │  prod_enc(GNN) + center_enc(GNN/Onehot)             │
  │  │  + inner_prod attention  │                           │
  │  └──────────────────────────┘                           │
  │                                                         │
  │  ┌──────────────────────────┐                           │
  │  │ ActiveProbCalc           │ → log P(template|prod)   │
  │  │  prod_enc(GNN) + tpl_enc(DeepSet/Onehot)            │
  │  │  + inner_prod attention  │                         

In [13]:
# ====== 5.2 模型训练示例 (微型数据) ======
# 展示 GraphPath 的完整训练过程

# 构造教学用的微型模板数据
# 酯化反应模板 (简化版)
DEMO_TEMPLATES = [
    '[C:1](=[O:2])[O:3][C:4]>>[C:1](=[O:2])[OH:3].[C:4][OH]',       # 酯键断裂
    '[c:1]-[c:2]>>[c:1][Br].[c:2][B]([OH])[OH]',                      # Suzuki偶联断裂
]

DEMO_PROD_CENTERS = [
    '[C](=[O])[O][C]',     # 酯键中心
    '[c]-[c]',              # C-C偶联中心
]

# 构造训练样本
demo_samples = [
    DataSample(
        prod='CC(=O)OC',                          # 乙酸甲酯
        center='[C](=[O])[O][C]',                  # 正确中心: 酯键
        template=DEMO_TEMPLATES[0],                # 正确模板: 酯键断裂
        neg_centers=['[c]-[c]'],                   # 负中心: C-C偶联
        neg_tpls=[DEMO_TEMPLATES[1]],              # 负模板: Suzuki
        reaction='CC(=O)O.CO>>CC(=O)OC',          # 正确反应
        neg_reactions=['CC(O)=O.CC>>CC(=O)OC']     # 负反应 (错误反应物)
    ),
    DataSample(
        prod='c1ccc(-c2ccccc2)cc1',                # 联苯
        center='[c]-[c]',                           # 正确中心: C-C偶联
        template=DEMO_TEMPLATES[1],                # 正确模板: Suzuki
        neg_centers=['[C](=[O])[O][C]'],           # 负中心: 酯键
        neg_tpls=[DEMO_TEMPLATES[0]],              # 负模板: 酯键断裂
        reaction='c1ccc(Br)cc1.OB(O)c1ccccc1>>c1ccc(-c2ccccc2)cc1',
        neg_reactions=['c1ccccc1.c1ccccc1>>c1ccc(-c2ccccc2)cc1']
    ),
]

# 初始化模型
model = GraphPath(
    embed_dim=32,
    att_type='inner_prod',
    retro_during_train=True,
    prod_cano_smarts=DEMO_PROD_CENTERS,
    tpl_enc='deepset'
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'GLN 模型参数总量: {total_params:,}')
print(f'  CenterProbCalc 参数: {sum(p.numel() for p in model.prod_center_predicate.parameters()):,}')
print(f'  ActiveProbCalc 参数: {sum(p.numel() for p in model.tpl_fwd_predicate.parameters()):,}')
print(f'  ReactionProbCalc 参数: {sum(p.numel() for p in model.reaction_predicate.parameters()):,}')

GLN 模型参数总量: 23,968
  CenterProbCalc 参数: 3,552
  ActiveProbCalc 参数: 13,504
  ReactionProbCalc 参数: 6,912


In [14]:
# ====== 5.3 训练循环演示 ======
# 对应 GLN 源码: gln/training/main.py → main_train()

optimizer = optim.Adam(model.parameters(), lr=0.001)

print('训练演示 (10 个 epoch，2 个样本/batch)')
print('=' * 50)

losses = []
for epoch in range(10):
    optimizer.zero_grad()
    loss = model(demo_samples)
    loss.backward()
    
    # 梯度裁剪 (GLN 默认使用 grad_clip=5.0)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    
    optimizer.step()
    losses.append(loss.item())
    
    if (epoch + 1) % 2 == 0:
        print(f'  Epoch {epoch+1:3d} | Loss: {loss.item():.4f}')

print()
print(f'训练完成! 初始损失: {losses[0]:.4f} → 最终损失: {losses[-1]:.4f}')
print()
print('损失趋势:', ' → '.join(f'{l:.2f}' for l in losses))

训练演示 (10 个 epoch，2 个样本/batch)
  Epoch   2 | Loss: 0.5190
  Epoch   4 | Loss: 0.0587
  Epoch   6 | Loss: 0.0062
  Epoch   8 | Loss: 0.0006
  Epoch  10 | Loss: 0.0001

训练完成! 初始损失: 2.2692 → 最终损失: 0.0001

损失趋势: 2.27 → 0.52 → 0.16 → 0.06 → 0.02 → 0.01 → 0.00 → 0.00 → 0.00 → 0.00


[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATIO

---

## 6. 推理流程端到端演示

GLN 的推理是一个**级联的 beam search** 过程：

```
输入: 产物 SMILES "CC(=O)Oc1ccccc1" (乙酸苯酯)
│
├── (1) 反应中心预测 (CenterProbCalc)
│   子结构匹配: 找出产物中所有可能的反应中心
│   GNN 评分: 对每个候选中心打分
│   选择 Top-K 中心
│
├── (2) 模板预测 (ActiveProbCalc)
│   对每个 Top-K 中心，找出关联的模板
│   GNN 评分: 对每个候选模板打分
│   综合分数: center_score + template_score
│   选择 Top-K 模板
│
├── (3) 模板应用
│   对每个 Top-K 模板，应用到产物上生成候选反应物
│   过滤无效结果
│
├── (4) 反应物验证 (ReactionProbCalc)
│   GNN 评分: 对每个候选反应物打分
│   最终分数: center_score + template_score + reaction_score
│
└── 输出: Top-K 预测结果（反应物、模板、分数）
```

### GLN 源码对应
- `gln/test/model_inference.py` → `RetroGLN.run()`

In [15]:
# ====== 6.1 推理类 RetroGLN ======
# 对应 GLN 源码: gln/test/model_inference.py → RetroGLN

from scipy.special import softmax as scipy_softmax


class RetroGLN:
    """
    GLN 逆合成推理器。
    
    对应 GLN 源码: gln/test/model_inference.py → RetroGLN
    
    推理步骤:
    1. 标准化产物 SMILES
    2. 子结构匹配找出候选反应中心
    3. CenterProbCalc 对中心评分
    4. 对 Top-K 中心，查找关联模板
    5. ActiveProbCalc 对模板评分
    6. 应用模板到产物，生成候选反应物
    7. ReactionProbCalc 对反应物评分
    8. 综合分数排序，返回 Top-K 结果
    """
    def __init__(self, model, prod_cano_smarts, unique_templates, tpl_of_center):
        """
        参数:
            model: 训练好的 GraphPath 模型
            prod_cano_smarts: 所有唯一产物中心 SMARTS 列表
            unique_templates: 所有唯一模板列表
            tpl_of_center: 中心 → 模板索引的映射
        """
        self.model = model
        self.model.eval()
        self.prod_cano_smarts = prod_cano_smarts
        self.unique_templates = unique_templates
        self.tpl_of_center = tpl_of_center
        
        # 预编译 SMARTS
        self.cached_smarts = [Chem.MolFromSmarts(sm) for sm in prod_cano_smarts]
    
    def cano_smiles(self, smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return smiles
        mol = Chem.RemoveHs(mol)
        [a.ClearProp('molAtomMapNumber') for a in mol.GetAtoms()]
        return Chem.MolToSmiles(mol)
    
    @torch.no_grad()
    def run(self, raw_prod, beam_size=10, topk=5):
        """
        执行逆合成预测。
        
        对应 GLN 源码: model_inference.py → RetroGLN.run()
        """
        cano_prod = self.cano_smiles(raw_prod)
        prod_mol_graph = SmilesMols.get_mol_graph(cano_prod)
        prod_mol = Chem.MolFromSmiles(cano_prod)
        if prod_mol is None:
            return None
        
        # ==== Step 1: 找出所有匹配的反应中心 ====
        matching_center_idx = []
        for i, sm in enumerate(self.cached_smarts):
            if sm is not None and prod_mol.HasSubstructMatch(sm):
                matching_center_idx.append(i)
        
        if not matching_center_idx:
            print(f'  [!] 无匹配中心')
            return None
        
        print(f'  [Step 1] 找到 {len(matching_center_idx)} 个匹配中心')
        
        # ==== Step 2: GNN 对中心评分 ====
        center_mols = [SmartsMols.get_mol_graph(self.prod_cano_smarts[i]) for i in matching_center_idx]
        center_scores = self.model.prod_center_predicate.inference(
            [prod_mol_graph], [center_mols]
        ).view(-1).cpu().numpy()
        
        # Top-K 中心
        top_center_idx = np.argsort(-center_scores)[:beam_size]
        print(f'  [Step 2] 中心评分 Top-{len(top_center_idx)}:')
        for idx in top_center_idx:
            center_smarts = self.prod_cano_smarts[matching_center_idx[idx]]
            print(f'    中心 [{matching_center_idx[idx]}] score={center_scores[idx]:.4f}: {center_smarts}')
        
        # ==== Step 3: 对 Top-K 中心查找关联模板并评分 ====
        tpl_with_scores = []
        list_of_list_tpls = []
        center_indices_for_tpls = []
        
        for i in top_center_idx:
            center_smarts = self.prod_cano_smarts[matching_center_idx[i]]
            if center_smarts not in self.tpl_of_center:
                continue
            tpls = self.tpl_of_center[center_smarts]
            if not tpls:
                continue
            list_of_list_tpls.append(tpls)
            center_indices_for_tpls.append(i)
        
        if not list_of_list_tpls:
            print(f'  [!] 无关联模板')
            return None
        
        tpl_scores = self.model.tpl_fwd_predicate.inference(
            [prod_mol_graph] * len(list_of_list_tpls), list_of_list_tpls
        ).view(-1).cpu().numpy()
        
        idx = 0
        for ci, tpls in zip(center_indices_for_tpls, list_of_list_tpls):
            for tpl in tpls:
                combined_score = center_scores[ci] + tpl_scores[idx]
                tpl_with_scores.append((combined_score, tpl))
                idx += 1
        
        tpl_with_scores.sort(key=lambda x: -x[0])
        print(f'  [Step 3] 模板评分: 共 {len(tpl_with_scores)} 个候选')
        
        # ==== Step 4: 应用模板生成候选反应物 ====
        list_of_list_reacts = []
        valid_tpls = []
        
        for score, tpl in tpl_with_scores[:beam_size]:
            # 用 RDKit 原生反应引擎应用模板
            parts = tpl.split('>')
            if len(parts) != 3:
                continue
            rxn_smarts = f'{parts[0]}>>{parts[2]}'
            try:
                rxn = AllChem.ReactionFromSmarts(rxn_smarts)
                if rxn is None:
                    continue
                results = rxn.RunReactants([prod_mol])
                outcomes = []
                seen = set()
                for products in results:
                    smi = '.'.join(sorted([Chem.MolToSmiles(p) for p in products]))
                    if smi not in seen:
                        outcomes.append(smi)
                        seen.add(smi)
                if outcomes:
                    list_of_list_reacts.append(outcomes)
                    valid_tpls.append((score, tpl))
            except:
                continue
        
        print(f'  [Step 4] 模板应用: {len(valid_tpls)} 个模板产生了有效反应物')
        
        if not valid_tpls:
            return {'reactants': [], 'template': [], 'scores': []}
        
        # ==== Step 5: GNN 对反应物评分 ====
        list_rxns = []
        for i, outcomes in enumerate(list_of_list_reacts):
            list_rxns.append([self.cano_smiles(r) + '>>' + cano_prod for r in outcomes])
        
        react_scores = self.model.reaction_predicate.inference(
            [prod_mol_graph] * len(valid_tpls), list_rxns
        ).view(-1).cpu().numpy()
        
        # 综合分数
        final_joint = []
        idx = 0
        for i, (prod_tpl_score, tpl) in enumerate(valid_tpls):
            for reacts in list_of_list_reacts[i]:
                combined = prod_tpl_score + react_scores[idx]
                final_joint.append((combined, tpl, reacts))
                idx += 1
        
        final_joint.sort(key=lambda x: -x[0])
        final_joint = final_joint[:topk]
        
        scores = scipy_softmax([t[0] for t in final_joint])
        
        print(f'  [Step 5] 最终排序: 输出 Top-{len(final_joint)} 预测')
        
        return {
            'reactants': [t[2] for t in final_joint],
            'template': [t[1] for t in final_joint],
            'scores': scores.tolist()
        }


print('RetroGLN 推理类定义完成')

RetroGLN 推理类定义完成


In [16]:
# ====== 6.2 端到端推理演示 ======

# 构建中心→模板映射（简化版）
tpl_of_center = {
    '[C](=[O])[O][C]': [DEMO_TEMPLATES[0]],    # 酯键中心 → 酯键断裂模板
    '[c]-[c]': [DEMO_TEMPLATES[1]],              # C-C偶联中心 → Suzuki模板
}

# 创建推理器
inferencer = RetroGLN(
    model=model,
    prod_cano_smarts=DEMO_PROD_CENTERS,
    unique_templates=DEMO_TEMPLATES,
    tpl_of_center=tpl_of_center
)

# ====== 演示推理 ======
test_products = [
    'CC(=O)OC',           # 乙酸甲酯 → 期望: 乙酸 + 甲醇
    'c1ccc(-c2ccccc2)cc1', # 联苯 → 期望: 溴苯 + 苯硼酸
]

print('='*70)
print('GLN 逆合成推理演示')
print('='*70)

for prod in test_products:
    print(f'\n产物: {prod}')
    print('-'*50)
    result = inferencer.run(prod, beam_size=5, topk=3)
    
    if result and result['reactants']:
        print(f'\n  预测结果:')
        for i, (reactants, template, score) in enumerate(
            zip(result['reactants'], result['template'], result['scores'])
        ):
            print(f'    Top-{i+1}: {reactants}')
            print(f'           模板: {template}')
            print(f'           分数: {score:.4f}')
    else:
        print('  [无预测结果]')
    print()

GLN 逆合成推理演示

产物: CC(=O)OC
--------------------------------------------------
  [Step 1] 找到 1 个匹配中心
  [Step 2] 中心评分 Top-1:
    中心 [0] score=11.8572: [C](=[O])[O][C]
  [Step 3] 模板评分: 共 1 个候选
  [Step 4] 模板应用: 1 个模板产生了有效反应物
  [Step 5] 最终排序: 输出 Top-1 预测

  预测结果:
    Top-1: CC(=O)O.CO
           模板: [C:1](=[O:2])[O:3][C:4]>>[C:1](=[O:2])[OH:3].[C:4][OH]
           分数: 1.0000


产物: c1ccc(-c2ccccc2)cc1
--------------------------------------------------
  [Step 1] 找到 1 个匹配中心
  [Step 2] 中心评分 Top-1:
    中心 [1] score=19.5511: [c]-[c]
  [Step 3] 模板评分: 共 1 个候选
  [Step 4] 模板应用: 1 个模板产生了有效反应物
  [Step 5] 最终排序: 输出 Top-1 预测

  预测结果:
    Top-1: Brc1ccccc1.OB(O)c1ccccc1
           模板: [c:1]-[c:2]>>[c:1][Br].[c:2][B]([OH])[OH]
           分数: 1.0000



[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[11:53:40] DEPRECATIO

---

## 7. 训练流程概览

### 训练数据生成

GLN 使用**多进程 + 队列**的方式持续生成训练样本（`data_gen.py`）：

```
┌─────────────┐   ┌─────────────┐   ┌─────────────┐
│  Worker 0   │   │  Worker 1   │   │  Worker 2   │
│  (Process)  │   │  (Process)  │   │  (Process)  │
│             │   │             │   │             │
│ 遍历反应数据 │   │ 遍历反应数据 │   │ 遍历反应数据 │
│ 采样正/负样本 │   │ 采样正/负样本 │   │ 采样正/负样本 │
└──────┬──────┘   └──────┬──────┘   └──────┬──────┘
       │                 │                 │
       └────────────┬────┘─────────────────┘
                    ↓
            ┌──────────────┐
            │ 共享队列 Queue │
            │ (max_qsize)  │
            └──────┬───────┘
                   ↓
            ┌──────────────┐
            │  主训练进程    │
            │ next(gen) ────→ 取样本
            │ optimizer.step │
            └──────────────┘
```

### 负采样策略

对每个训练样本，生成三种负样本：

| 负样本类型 | 来源 | 说明 |
|-----------|------|------|
| 负中心 | 同一产物的其他匹配中心 | 产物匹配但非正确的反应中心 |
| 负模板 | 正确中心和负中心下的其他模板 | 同一中心位置的错误模板 |
| 负反应物 | 错误模板应用到产物的结果 | 模板应用产生的错误反应物 |

### 完整训练流程

```python
# GLN 源码: gln/training/main.py (简化版)
DataInfo.init(dropbox, args)              # 加载所有预处理数据
load_bin_feats(dropbox, args)             # 加载二进制分子图
graph_path = GraphPath(args).to(DEVICE)   # 初始化模型
optimizer = Adam(graph_path.parameters(), lr=args.learning_rate)

train_gen = data_gen(num_workers, worker_softmax, [args])

for epoch in range(num_epochs):
    for it in range(iters_per_val):
        samples = [next(train_gen) for _ in range(batch_size)]
        optimizer.zero_grad()
        loss = graph_path(samples)
        loss.backward()
        clip_grad_norm_(graph_path.parameters(), max_norm=grad_clip)
        optimizer.step()
    
    # 保存检查点
    if epoch % epochs2save == 0:
        save_model(graph_path, epoch)
```

### 评估指标

GLN 在测试集上计算 **Top-K 精确匹配准确率**：
- Top-1: 第一个预测是否正确
- Top-3: 前三个预测中是否有正确的
- Top-5: 前五个预测中是否有正确的
- Top-10: 前十个预测中是否有正确的

在 Schneider50k 数据集上，GLN 论文报告的性能：

| 指标 | 准确率 |
|------|-------|
| Top-1 | 52.5% |
| Top-3 | 69.0% |
| Top-5 | 75.6% |
| Top-10 | 83.7% |

In [17]:
# ====== 7.1 训练循环演示（使用前面定义的模型与数据） ======
# 对应 GLN 源码: gln/training/main.py → main_train()

# 重新初始化模型（避免受前面训练影响）
train_model = GraphPath(
    embed_dim=32,
    att_type='inner_prod',
    retro_during_train=True,
    prod_cano_smarts=DEMO_PROD_CENTERS,
    tpl_enc='deepset'
).to(DEVICE)

optimizer = torch.optim.Adam(train_model.parameters(), lr=1e-3)

print("=" * 60)
print("GLN 训练循环演示（Section 7）")
print("=" * 60)
print(f"模型参数量: {sum(p.numel() for p in train_model.parameters()):,}")
print(f"训练样本数: {len(demo_samples)}")
print(f"优化器: Adam (lr=1e-3)")
print(f"梯度裁剪: max_norm=5.0")
print()

n_epochs = 20
losses = []

for epoch in range(n_epochs):
    optimizer.zero_grad()
    
    # ---- 前向传播 ----
    # GraphPath.forward(samples) 内部计算:
    #   loss = -mean(log P(center|prod)) 
    #        - mean(log P(template|prod))
    #        - mean(log P(reactants|prod))
    loss = train_model(demo_samples)
    
    # ---- 反向传播 ----
    loss.backward()
    
    # 梯度裁剪（GLN 默认 max_norm=5.0）
    grad_norm = torch.nn.utils.clip_grad_norm_(train_model.parameters(), max_norm=5.0)
    
    # ---- 参数更新 ----
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{n_epochs} | "
              f"Loss: {loss.item():.4f} | "
              f"Grad Norm: {grad_norm:.4f}")

print()
print("-" * 60)
print(f"训练完成! 初始损失: {losses[0]:.4f} → 最终损失: {losses[-1]:.4f}")
print(f"损失下降: {losses[0] - losses[-1]:.4f}")
print()
print("损失变化趋势:")
print("  " + " → ".join(f"{l:.2f}" for l in losses[::4]))
print()
print("说明:")
print("  实际训练中 GLN 使用多进程 data_gen() 持续生成 DataSample，")
print("  每个 batch 包含多个样本，经过数千次迭代优化。")
print("  上面的演示使用 2 个固定的微型样本，仅展示训练框架。")

GLN 训练循环演示（Section 7）
模型参数量: 23,968
训练样本数: 2
优化器: Adam (lr=1e-3)
梯度裁剪: max_norm=5.0

  Epoch   1/20 | Loss: 2.4206 | Grad Norm: 302.8876
  Epoch   5/20 | Loss: 0.1573 | Grad Norm: 2.4235
  Epoch  10/20 | Loss: 0.0129 | Grad Norm: 0.2846
  Epoch  15/20 | Loss: 0.0018 | Grad Norm: 0.0516
  Epoch  20/20 | Loss: 0.0004 | Grad Norm: 0.0146

------------------------------------------------------------
训练完成! 初始损失: 2.4206 → 最终损失: 0.0004
损失下降: 2.4202

损失变化趋势:
  2.42 → 0.16 → 0.02 → 0.00 → 0.00

说明:
  实际训练中 GLN 使用多进程 data_gen() 持续生成 DataSample，
  每个 batch 包含多个样本，经过数千次迭代优化。
  上面的演示使用 2 个固定的微型样本，仅展示训练框架。


---

## 8. 总结

### 本 Notebook 回顾

在本教程中，我们从零开始拆解了 **GLN（Graph Logic Network）** 的完整模型架构：

| 章节 | 内容 | 关键组件 |
|------|------|---------|
| §1 | 分子图特征编码 | `get_atom_features()`, `get_bond_features()`, `MolGraph`, `MolGraphPool` |
| §2 | GNN 编码器 | `MLP`, `ReadoutNet`, `EmbedMeanField` |
| §3 | Jagged Softmax | `jagged_log_softmax()` — 变长候选列表上的 softmax |
| §4 | 三个概率头 | `CenterProbCalc`(中心), `ActiveProbCalc`(模板), `ReactionProbCalc`(反应) |
| §5 | 主模型 GraphPath | `DataSample`, `GraphPath` — 联合三个概率头的统一模型 |
| §6 | 推理管线 | `RetroGLN` — 从产物 SMILES 到逆合成预测的完整流程 |
| §7 | 训练流程 | 多进程数据生成，负采样策略，联合损失优化 |

### GLN 的核心思想

**Graph Logic Network** 的创新点在于将逆合成问题分解为**条件概率链**：

$$P(\text{reactants} | \text{product}) = \sum_{c} \sum_{t} P(c|p) \cdot P(t|c,p) \cdot P(r|t,p)$$

- **化学可解释性**：每一步对应明确的化学含义（断键位置 → 反应模板 → 反应物）
- **逻辑约束**：模板只在匹配的中心上评分，保证化学合法性
- **端到端训练**：三个子问题共享 GNN 编码器，联合优化

### 教学系列导航

```
📂 2.1.1.gln/
├── 1_环境配置.ipynb    ← 依赖安装与代码结构概览
├── 2_数据处理.ipynb    ← 数据预处理全流程拆解
└── 3_模型展示.ipynb    ← 模型架构与推理/训练（本 Notebook）
```

### 参考文献

1. **Dai, H., Li, C., Coley, C., Dai, B., & Song, L.** (2019). *Retrosynthesis Prediction with Conditional Graph Logic Network.* NeurIPS 2019. [arXiv:2001.01408](https://arxiv.org/abs/2001.01408)
2. **Coley, C. W., et al.** (2017). *Computer-Assisted Retrosynthesis Based on Molecular Similarity.* ACS Central Science.
3. **RDChiral**: Thakkar, A. et al. — 立体化学感知的模板提取与应用工具

---

✅ **恭喜完成 GLN 模型架构全流程的学习！**